# EC の商品名からブランド・色・サイズ・素材・容量を取り出す（辞書と正規表現、BERT、LLM）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/17_ner/product_attribute_extraction.ipynb)

NLP クックブック 17-14 ／ 固有表現抽出

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

通販サイトの商品名から、ブランド・色・サイズ・素材・容量といった属性の値を取り出し、「Mサイズ」と「M」、「ブラック」と「黒」のような表記の違いをそろえて、商品マスタの値と突き合わせたい。辞書と正規表現、BERT のトークン分類、LLM による JSON 抽出を、属性ごとの適合率・再現率で比べる。

In [1]:
import random, re, unicodedata
import numpy as np
import pandas as pd

ATTRS = ["brand", "color", "size", "material", "capacity"]
# 評価用の商品名 (このレシピのために作ったもの。実在のブランド名や商品名・型番も含むが、属性の組み合わせは実際の商品の仕様とは限らない)
# [表記|属性] が正解。表記と正規化した値が違うときは [表記|属性=正規化した値] と書く
TEST = """\
【送料無料】[NIKE|brand=ナイキ] エア ズーム ペガサス ランニングシューズ メンズ [ブラック|color=黒] [26.5cm|size]
[ユニクロ|brand] クルーネックT 半袖 [ホワイト|color=白] [Mサイズ|size=M] [綿|material]60% [ポリエステル|material]40%
[サーモス|brand] 真空断熱ケータイマグ [500ml|capacity] [ステンレス|material] [ネイビー|color=紺] JNL-506
[THERMOS|brand=サーモス] 真空断熱タンブラー [0.4L|capacity=400ml] [シルバー|color=銀]
[象印|brand] [ステンレス|material]マグ シームレスせん [480mL|capacity=480ml] [ピンク|color]
[アイリスオーヤマ|brand] 電気ケトル [1.0L|capacity=1000ml] [ホワイト|color=白] 空焚き防止
[タイガー|brand] 魔法瓶 [ステンレス|material]ボトル [600ml|capacity] [マットブラック|color=黒]
[ワークマン|brand] 防水防寒ジャケット [LL|size=XL] [ブラック|color=黒]
[ニトリ|brand] [綿|material]100% ボックスシーツ [シングル|size] [グレー|color=灰]
【楽天1位】[無印良品|brand] [麻|material]平織 ベッドシーツ [セミダブル|size] [ネイビー|color=紺]
[エレコム|brand] USBメモリ [64GB|capacity] USB3.2 [ブルー|color=青]
[SanDisk|brand=サンディスク] microSDXCカード [128GB|capacity] UHS-I
[バッファロー|brand] 外付けHDD [2TB|capacity] [黒|color] テレビ録画対応
[Anker|brand=アンカー] モバイルバッテリー [10000mAh|capacity] [ブラック|color=黒]
[アディダス|brand] スタンスミス スニーカー [ホワイト|color=白]×[グリーン|color=緑] [25.0cm|size=25cm]
[New Balance|brand=ニューバランス] 996 スニーカー [グレー|color=灰] [27cm|size]
[コールマン|brand] アウトドアチェア [スチール|material]フレーム [グリーン|color=緑]
[山善|brand] 扇風機 リビング扇 [ホワイト|color=白] リモコン付き
[ロジクール|brand] ワイヤレスマウス 静音 [グラファイト|color=灰]
【まとめ買い】天然水 [2L|capacity=2000ml]×9本
[パナソニック|brand] ドライヤー ナノケア [ディープネイビー|color=紺]
[カシオ|brand] 腕時計 スタンダード [ステンレス|material]バンド [シルバー|color=銀]
メンズ ビジネスシューズ [本革|material=革] [ブラウン|color=茶] [26.0cm|size=26cm] 幅広 3E
レディース [ウール|material]100% ニットカーディガン [ベージュ|color] [フリーサイズ|size=フリー]
綿棒 200本入り 抗菌 個包装
ホワイトニング 歯磨き粉 薬用 [90g|capacity]
L字デスク [木|material]製 幅120cm [ブラウン|color=茶]
麻婆豆腐の素 中辛 [150g|capacity]
[PUレザー|material=合成皮革] トートバッグ レディース 大容量 A4対応 [キャメル|color=茶]
[STANLEY|brand=スタンレー] クラシック真空ボトル [1.0L|capacity=1000ml] [グリーン|color=緑]
[mont-bell|brand=モンベル] ダウンジャケット [ナイロン|material] [XL|size] [レッド|color=赤]
キッズ Tシャツ [130cm|size] [イエロー|color=黄] [コットン|material=綿]100%
[ユニクロ|brand] ウルトラライトダウン [ネイビー|color=紺] [L|size]
[無印良品|brand] [ポリプロピレン|material]収納ケース 引出式 深型
[Dyson|brand=ダイソン] コードレス掃除機 V8 [シルバー|color=銀]
[ナイキ|brand] ドライフィット ハーフパンツ [S|size] [ブラック|color=黒]/[ホワイト|color=白]
[アイリスオーヤマ|brand] 布団乾燥機 カラリエ [ピンク|color]
[象印|brand] 炊飯器 [5.5合|capacity] 極め炊き [黒|color]
[タイガー|brand] 電気ポット [2.2L|capacity=2200ml] 蒸気レス [ブラウン|color=茶]
国産 [ヒノキ|material] まな板 [L|size]
[エレコム|brand] スマホケース iPhone15 [シリコン|material] [ブラック|color=黒]
レインコート 自転車 [LL|size=XL] 男女兼用 [カーキ|color]
[ニトリ|brand] 遮光カーテン 幅100×丈178cm 2枚組 [ベージュ|color]
[パナソニック|brand] 充電池 エネループ 単3形 4本パック [1900mAh|capacity]
【2個セット】[ガラス|material]製 保存容器 [800ml|capacity] 耐熱
[カシオ|brand] 電子辞書 エクスワード [ホワイト|color=白]
メンズ チノパン ストレッチ [W82|size] [ベージュ|color]
[THERMOS|brand=サーモス] フライパン [26cm|size] IH対応
[アディダス|brand] スポーツバッグ [ポリエステル|material] [30L|capacity=30000ml] [ネイビー|color=紺]
[ワークマン|brand] 作業用 手袋 [M|size] 滑り止め [グレー|color=灰]
[Coleman|brand=コールマン] ソフトクーラー [15L|capacity=15000ml] [ブルー|color=青]
[ロジクール|brand] キーボード 日本語配列 [オフホワイト|color=白]
子供用 長靴 [18cm|size] [レッド|color=赤] [ゴム|material]製
[UNIQLO|brand=ユニクロ] ヒートテック タイツ [XXL|size] [黒|color]
[アンカー|brand] USB充電器 65W [ホワイト|color=白]
[山善|brand] こたつ 長方形 [天然木|material=木] 105×75cm
[無印良品|brand] [オーガニックコットン|material=綿] オックスシャツ [S|size] [サックスブルー|color=青]
[象印|brand] 水筒 [0.36L|capacity=360ml] [ステンレス|material]マグ [ミントブルー|color=青]
USB-C ハブ 7in1 [スペースグレー|color=灰] [アルミ|material]
[ナイキ|brand] リュック [25L|capacity=25000ml] [ブラック|color=黒]
大きいサイズ メンズ ポロシャツ [3L|size=XXL] [ネイビー|color=紺] [綿|material]100%"""

MARK = re.compile(r"\[([^\[\]|]+)\|(\w+)(?:=([^\]]+))?\]")

def parse(line):
    """[表記|属性=値] の印を外して、(本文, [(開始, 終了, 属性, 正規化した値), ...]) を返す"""
    text, spans, pos = "", [], 0
    for m in MARK.finditer(line):
        text += line[pos:m.start()]
        spans.append((len(text), len(text) + len(m[1]), m[2], m[3] or m[1]))
        text += m[1]
        pos = m.end()
    return text + line[pos:], spans

test = [parse(line) for line in TEST.split("\n")]
print(len(test), "件。属性ごとの正解の数:", pd.Series([a for _, g in test for *_, a, _ in g]).value_counts().to_dict())
print("表記と正規化した値が違う正解:", sum(s != c for line in TEST.split("\n") for s, _, c in MARK.findall(line) if c), "個")
print(test[1])

61 件。属性ごとの正解の数: {'color': 52, 'brand': 45, 'material': 26, 'size': 22, 'capacity': 21}
表記と正規化した値が違う正解: 76 個
('ユニクロ クルーネックT 半袖 ホワイト Mサイズ 綿60% ポリエステル40%', [(0, 4, 'brand', 'ユニクロ'), (16, 20, 'color', '白'), (21, 25, 'size', 'M'), (26, 27, 'material', '綿'), (31, 37, 'material', 'ポリエステル')])


In [2]:
# 値の正規化: 商品マスタの値 (左) と、その別表記 (右)。辞書による抽出にも同じ表を使う
BRANDS = {"ナイキ": ["NIKE"], "アディダス": ["adidas"], "ユニクロ": ["UNIQLO"], "無印良品": ["MUJI"], "サーモス": ["THERMOS"],
          "象印": ["ZOJIRUSHI"], "タイガー": ["TIGER"], "アイリスオーヤマ": ["IRIS OHYAMA"], "パナソニック": ["Panasonic"],
          "エレコム": ["ELECOM"], "サンディスク": ["SanDisk"], "コールマン": ["Coleman"], "アンカー": ["Anker"],
          "ニューバランス": ["New Balance"]}
COLORS = {"黒": ["ブラック", "BLACK", "黒色"], "白": ["ホワイト", "WHITE", "白色"], "赤": ["レッド", "RED"], "青": ["ブルー", "BLUE"],
          "紺": ["ネイビー", "NAVY"], "灰": ["グレー", "グレイ", "GRAY"], "茶": ["ブラウン", "BROWN"], "緑": ["グリーン", "GREEN"],
          "黄": ["イエロー", "YELLOW"], "銀": ["シルバー", "SILVER"], "ピンク": ["PINK"], "ベージュ": [], "カーキ": []}
MATERIALS = {"綿": ["コットン", "cotton"], "革": ["本革", "レザー", "牛革"], "合成皮革": ["合皮", "PUレザー", "フェイクレザー"],
             "麻": ["リネン"], "ウール": ["羊毛"], "木": ["天然木", "木製"], "ステンレス": [], "ポリエステル": [], "ナイロン": [],
             "ガラス": [], "シリコン": ["シリコーン"]}
TABLES = {"brand": BRANDS, "color": COLORS, "material": MATERIALS}
ALIAS = {attr: {unicodedata.normalize("NFKC", a).lower(): canon for canon, aliases in table.items() for a in [canon, *aliases]}
         for attr, table in TABLES.items()}
SIZE_ALIAS = {"LL": "XL", "2L": "XL", "3L": "XXL", "4L": "XXXL", "F": "フリー"}
UNITS = {"ml": ("ml", 1), "ミリリットル": ("ml", 1), "l": ("ml", 1000), "リットル": ("ml", 1000), "g": ("g", 1), "kg": ("g", 1000),
         "gb": ("GB", 1), "tb": ("TB", 1), "mah": ("mAh", 1), "合": ("合", 1)}

def normalize(attr, value):
    v = unicodedata.normalize("NFKC", str(value)).strip()   # 全角英数字→半角、「ℓ」→「l」など
    if attr in ALIAS:
        alias = ALIAS[attr]
        if v.lower() in alias:
            return alias[v.lower()]
        hits = [a for a in alias if a in v.lower()]            # 「マットブラック」→「ブラック」→「黒」
        return alias[max(hits, key=len)] if hits else v
    if attr == "size":
        v = re.sub(r"サイズ|\s", "", v.upper())
        m = re.fullmatch(r"(\d+(?:\.\d+)?)CM", v)
        return f"{float(m[1]):g}cm" if m else SIZE_ALIAS.get(v, v)
    if attr == "capacity":
        m = re.search(r"(\d+(?:\.\d+)?)\s*(mah|ml|ミリリットル|リットル|l|kg|g|gb|tb|合)", v.lower())
        if m:
            unit, k = UNITS[m[2]]
            return f"{float(m[1]) * k:g}{unit}"
    return v

for attr, v in [("size", "Mサイズ"), ("size", "Ｍ"), ("size", "3L"), ("size", "25.0cm"), ("color", "マットブラック"),
                ("color", "BLACK"), ("material", "PUレザー"), ("capacity", "1.0L"), ("capacity", "480mL"), ("capacity", "５００ｍｌ")]:
    print(f"{attr:9}{v:10} → {normalize(attr, v)}")

size     Mサイズ       → M
size     Ｍ          → M
size     3L         → XXL
size     25.0cm     → 25cm
color    マットブラック    → 黒
color    BLACK      → 黒
material PUレザー      → 合成皮革
capacity 1.0L       → 1000ml
capacity 480mL      → 480ml
capacity ５００ｍｌ      → 500ml


In [3]:
def evaluate(preds, name, use_norm=True):
    """preds: 商品ごとの [(属性, 値), ...]。(属性, 正規化した値) の集合で正解と比べ、属性ごとの適合率・再現率・F1 を返す"""
    counts = {a: np.zeros((len(test), 3)) for a in ATTRS}      # 商品ごとの (正しい予測, 誤った予測, 見落とし)
    for i, ((text, gold), pred) in enumerate(zip(test, preds)):
        g = {(a, c) for *_, a, c in gold}
        p = {(a, normalize(a, v) if use_norm else str(v)) for a, v in pred if a in ATTRS and str(v).strip()}
        for a in ATTRS:
            ga, pa = {x for x in g if x[0] == a}, {x for x in p if x[0] == a}
            counts[a][i] = len(ga & pa), len(pa - ga), len(ga - pa)
    def prf(tp, fp, fn):
        p, r = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
        return p, r, 2 * p * r / max(p + r, 1e-9)
    rows = {a: prf(*counts[a].sum(0)) for a in ATTRS}
    total = sum(counts.values())
    rows["全体 (micro)"] = prf(*total.sum(0))
    rng = np.random.default_rng(0)                            # 商品を復元抽出して micro F1 の 95% 信頼区間を出す
    boot = [prf(*total[rng.integers(0, len(test), len(test))].sum(0))[2] for _ in range(1000)]
    df = pd.DataFrame(rows, index=["P", "R", "F1"]).T.round(2)
    df.columns = pd.MultiIndex.from_product([[name], df.columns])
    return df, np.percentile(boot, [2.5, 97.5])

In [4]:
# 方法 1: 辞書と正規表現
def dict_regex(table_alias):
    words = sorted(table_alias, key=len, reverse=True)       # 長い語を先に (「PUレザー」を「レザー」より優先)
    return re.compile("|".join(re.escape(w) for w in words), re.IGNORECASE)

DICT_RE = {attr: dict_regex(alias) for attr, alias in ALIAS.items()}
SIZE_RE = re.compile(r"(?<![A-Za-z0-9.])(?:XXL|XL|LL|[2-4]L|S|M|L|フリー)(?:サイズ)?(?![A-Za-z0-9])|\d{2,3}(?:\.\d)?cm|W\d{2}")
CAPACITY_RE = re.compile(r"\d+(?:\.\d+)?(?:mAh|ml|mL|GB|TB|kg|g|L|合)(?![A-Za-z])")

def extract_rules(text):
    text = unicodedata.normalize("NFKC", text)
    out = [(attr, m[0]) for attr, pat in DICT_RE.items() for m in pat.finditer(text)]
    out += [("size", m[0]) for m in SIZE_RE.finditer(text)] + [("capacity", m[0]) for m in CAPACITY_RE.finditer(text)]
    return out

rule_preds = [extract_rules(t) for t, _ in test]
for (t, _), p in list(zip(test, rule_preds))[24:29]:
    print(t, "→", p)
results = {}
for use_norm in (False, True):
    df, ci = evaluate(rule_preds, "辞書+正規表現", use_norm)
    print(f"正規化{'あり' if use_norm else 'なし'}: micro F1 {df.iloc[-1, 2]:.2f} (95% CI {ci[0]:.2f}〜{ci[1]:.2f})")
    if use_norm:
        results["辞書+正規表現"] = (df, ci)
df

綿棒 200本入り 抗菌 個包装 → [('material', '綿')]
ホワイトニング 歯磨き粉 薬用 90g → [('color', 'ホワイト'), ('capacity', '90g')]
L字デスク 木製 幅120cm ブラウン → [('color', 'ブラウン'), ('material', '木製'), ('size', 'L'), ('size', '120cm')]
麻婆豆腐の素 中辛 150g → [('material', '麻'), ('capacity', '150g')]
PUレザー トートバッグ レディース 大容量 A4対応 キャメル → [('material', 'PUレザー')]
正規化なし: micro F1 0.45 (95% CI 0.37〜0.51)
正規化あり: micro F1 0.89 (95% CI 0.84〜0.92)


辞書+正規表現            
                 P     R    F1
brand         1.00  0.69  0.82
color         0.98  0.96  0.97
size          0.80  0.91  0.85
material      0.91  0.81  0.86
capacity      0.86  0.90  0.88
全体 (micro)    0.93  0.85  0.89

In [5]:
# 方法 2 の準備: BERT の学習データをテンプレートから自動で作る (人手のアノテーションなし)
rng = random.Random(0)
KANA = list("アイウエオカキクケコサシスセソタチツテトナニヌネノハヒフヘホマミムメモヤユヨラリルレロワン")

def rand_brand():
    """辞書のブランドに加えて、ランダムな名前も混ぜる (辞書にないブランドにも対応できるように)"""
    if rng.random() < 0.5:
        canon = rng.choice(list(BRANDS))
        return rng.choice([canon, *BRANDS[canon]])
    if rng.random() < 0.6:
        return "".join(rng.choices(KANA, k=rng.randint(2, 5))) + rng.choice(["", "", "ー", "ン"])
    return "".join(rng.choices("abcdefghijklmnoprstuvwxyz", k=rng.randint(3, 7))).capitalize()

def pick(table):
    canon = rng.choice(list(table))
    return rng.choice([canon, *table[canon]])

SLOTS = {
    "color": lambda: rng.choice(["", "", "マット", "ダーク", "ライト"]) + pick(COLORS),
    "size": lambda: rng.choice(["S", "M", "L", "XL", "LL", "3L", "XXL"]) + rng.choice(["", "", "サイズ"]),
    "shoe": lambda: f"{rng.choice(range(22, 30))}{rng.choice(['', '.0', '.5'])}cm",
    "bed": lambda: rng.choice(["シングル", "セミダブル", "ダブル"]),
    "material": lambda: pick(MATERIALS),
    "ml": lambda: rng.choice([f"{rng.choice([300, 350, 480, 500, 600, 750, 800])}{rng.choice(['ml', 'mL'])}",
                              f"{rng.choice(['0.5', '1.0', '1.2', '1.5', '2', '2.2'])}L"]),
    "gb": lambda: f"{rng.choice([16, 32, 64, 128, 256, 512])}GB" if rng.random() < 0.8 else f"{rng.choice([1, 2, 4])}TB",
    "mah": lambda: f"{rng.choice([5000, 10000, 20000])}mAh",
}
LABEL = {"shoe": "size", "bed": "size", "ml": "capacity", "gb": "capacity", "mah": "capacity"}
CATEGORIES = [
    (["Tシャツ", "パーカー", "ジャケット", "シャツ", "パンツ", "ニット", "ポロシャツ", "スカート"], ["color", "size", "material"]),
    (["スニーカー", "ランニングシューズ", "サンダル", "ブーツ"], ["color", "shoe"]),
    (["水筒", "タンブラー", "マグボトル", "電気ケトル"], ["ml", "color", "material"]),
    (["USBメモリ", "SDカード", "外付けSSD"], ["gb", "color"]),
    (["モバイルバッテリー"], ["mah", "color"]),
    (["シーツ", "布団カバー", "枕カバー"], ["color", "bed", "material"]),
    (["ワイヤレスイヤホン", "電動歯ブラシ", "デスクライト", "財布", "ショルダーバッグ", "スマートウォッチ", "加湿器"], ["color", "material"]),
]
NOISE = ["", "", "", "【送料無料】", "【公式】", "ポイント10倍 ", "2024年モデル ", "【あす楽】"]
# 属性ではない語 (宣伝文句や機能) も混ぜておかないと、属性以外の語まで属性として取り出すようになる
EXTRA = ["メンズ", "レディース", "キッズ", "大容量", "軽量", "撥水", "保温", "保冷", "速乾", "UVカット", "おしゃれ", "ギフト", "人気",
         "新作", "限定", "日本製", "洗える", "折りたたみ", "コンパクト", "3枚セット", "母の日", "プレゼント", "通勤", "アウトドア"]

def make_example():
    nouns, slots = rng.choice(CATEGORIES)
    parts = [(rng.choice(NOISE), None)]
    if rng.random() < 0.85:
        parts += [(rand_brand(), "brand"), (" ", None)]
    parts += [(rng.choice(nouns), None)]
    for slot in rng.sample(slots, len(slots)):
        if rng.random() < 0.8:
            value, label = SLOTS[slot](), LABEL.get(slot, slot)
            parts += [(" ", None), (value, label)]
            if label == "material":
                parts += [(rng.choice(["", "100%", "素材", "製"]), None)]
    for _ in range(rng.choice([0, 1, 1, 2, 3])):
        gaps = [k for k, (s, _) in enumerate(parts) if s == " "] + [len(parts)]
        parts.insert(rng.choice(gaps), (f" {rng.choice(EXTRA)} ", None))
    text, spans = "", []
    for s, label in parts:
        if not label and text.endswith(" "):
            s = s.lstrip()                                     # 空白が続かないようにする
        if label:
            spans.append((len(text), len(text) + len(s), label))
        text += s
    return text.rstrip(), spans

train = [make_example() for _ in range(3000)]
for t, s in train[:3]:
    print(t, [(t[a:b], l) for a, b, l in s])

2024年モデル THERMOS アウトドア プレゼント 財布 UVカット 麻素材 シルバー [('THERMOS', 'brand'), ('麻', 'material'), ('シルバー', 'color')]
【あす楽】象印 電気ケトル アウトドア 綿製 2L [('象印', 'brand'), ('綿', 'material'), ('2L', 'capacity')]
ユナエー 電気ケトル 600mL ガラス製 [('ユナエー', 'brand'), ('600mL', 'capacity'), ('ガラス', 'material')]


In [6]:
import torch
from datasets import Dataset
from transformers import (AutoModelForTokenClassification, AutoTokenizer, DataCollatorForTokenClassification,
                          Trainer, TrainingArguments, pipeline)

torch.manual_seed(0)
BF16 = torch.cuda.get_device_capability()[0] >= 8
labels = ["O"] + [f"{p}-{a}" for a in ATTRS for p in ("B", "I")]
label2id = {l: i for i, l in enumerate(labels)}
model_name = "xlm-roberta-base"   # MIT ライセンス。文字位置 (offset mapping) を返す高速トークナイザーがある
tokenizer = AutoTokenizer.from_pretrained(model_name)

def encode(example):
    enc = tokenizer(example["text"], truncation=True, max_length=96, return_offsets_mapping=True)
    tags = []
    for tok, (s, e) in zip(enc.tokens(), enc["offset_mapping"]):
        tag = "O"
        if s == e:
            tags.append(-100)                                # 特殊トークンは損失の計算から除く
            continue
        if tok != "▁":                                       # 単独の「▁」は次の文字と同じ位置を返すので O にする (この章の「BERT をファインチューニングして日本語の固有表現抽出器を作る」を参照)
            for start, end, attr in zip(example["starts"], example["ends"], example["attrs"]):
                if s < end and e > start:
                    tag = ("B-" if s <= start else "I-") + attr
                    break
        tags.append(label2id[tag])
    enc["labels"] = tags
    enc.pop("offset_mapping")
    return enc

ds = Dataset.from_list([{"text": t, "starts": [a for a, _, _ in s], "ends": [b for _, b, _ in s], "attrs": [l for *_, l in s]}
                        for t, s in train]).map(encode, remove_columns=["text", "starts", "ends", "attrs"])
model = AutoModelForTokenClassification.from_pretrained(model_name, num_labels=len(labels),
                                                        id2label=dict(enumerate(labels)), label2id=label2id)
args = TrainingArguments(output_dir="attr-out", num_train_epochs=3, learning_rate=5e-5, warmup_steps=30,
                         per_device_train_batch_size=32, save_strategy="no", logging_steps=50, report_to="none",
                         seed=0, bf16=BF16, fp16=not BF16)
trainer = Trainer(model=model, args=args, train_dataset=ds, data_collator=DataCollatorForTokenClassification(tokenizer))
trainer.train()   # T4 で 1〜2 分

Step,Training Loss
50,1.281337
100,0.090280
150,0.008570
200,0.004935
250,0.003303


TrainOutput(global_step=282, training_loss=0.24635718241069757, metrics={'train_runtime': 28.5306, 'train_samples_per_second': 315.451, 'train_steps_per_second': 9.884, 'total_flos': 150392602480656.0, 'train_loss': 0.24635718241069757, 'epoch': 3.0})

In [7]:
# 方法 2: BERT (XLM-R) のトークン分類で、評価用の商品名から属性を取り出す
ner = pipeline("token-classification", model=model, tokenizer=tokenizer, aggregation_strategy="simple", device=0)
raw = ner([t for t, _ in test])
bert_preds = [[(e["entity_group"], t[e["start"]:e["end"]].strip()) for e in ents] for (t, _), ents in zip(test, raw)]
for (t, _), ents in list(zip(test, raw))[24:29]:
    print(t, "→", [(e["entity_group"], t[e["start"]:e["end"]].strip(), round(float(e["score"]), 2)) for e in ents])
results["BERT"] = evaluate(bert_preds, "BERT")
print(f"micro F1 95% CI: {results['BERT'][1][0]:.2f}〜{results['BERT'][1][1]:.2f}")
# 確信度 (score) の低いものを捨てると、適合率が上がり再現率が下がる。閾値は本来、評価データとは別の開発データで決める
for th in [0.5, 0.7, 0.9]:
    kept = [[(a, v) for (a, v), e in zip(p, ents) if e["score"] >= th] for p, ents in zip(bert_preds, raw)]
    print(f"score >= {th}:", dict(zip(["P", "R", "F1"], evaluate(kept, "BERT")[0].iloc[-1])))
results["BERT"][0]

綿棒 200本入り 抗菌 個包装 → [('brand', '綿', 0.64), ('capacity', '200本', 0.84)]
ホワイトニング 歯磨き粉 薬用 90g → [('brand', 'ホワイト', 0.86), ('capacity', '90g', 0.99)]
L字デスク 木製 幅120cm ブラウン → [('brand', 'L字', 0.65), ('material', '木', 1.0), ('material', '幅', 0.94), ('size', '120cm', 0.8), ('color', 'ブラウン', 1.0)]
麻婆豆腐の素 中辛 150g → [('brand', '麻婆豆腐', 1.0), ('material', '素', 0.92), ('material', '中辛', 0.98), ('capacity', '150g', 0.97)]
PUレザー トートバッグ レディース 大容量 A4対応 キャメル → [('brand', 'PUレザー', 1.0), ('size', 'A', 0.98), ('material', '4', 0.74), ('material', 'キャメル', 0.99)]
micro F1 95% CI: 0.67〜0.79
score >= 0.5: {'P': 0.64, 'R': 0.89, 'F1': 0.74}
score >= 0.7: {'P': 0.7, 'R': 0.89, 'F1': 0.78}
score >= 0.9: {'P': 0.77, 'R': 0.87, 'F1': 0.82}


BERT            
               P     R    F1
brand       0.66  0.91  0.77
color       0.89  0.92  0.91
size        0.51  0.95  0.67
material    0.38  0.85  0.52
capacity    0.67  0.76  0.71
全体 (micro)  0.62  0.89  0.73

In [8]:
# 方法 3: LLM に JSON で答えさせる (sarashina2.2-3b-instruct、MIT ライセンス)
import gc, json
import sentencepiece as spm
from huggingface_hub import hf_hub_download
from tokenizers import models
from transformers import AutoModelForCausalLM

del trainer, model, ner
gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"
llm_tok = AutoTokenizer.from_pretrained(name, padding_side="left")
# transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる。
# 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す (特殊トークンやチャットテンプレートはそのまま)
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
llm_tok.backend_tokenizer.model = models.Unigram(
    [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
llm = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16 if BF16 else torch.float16, device_map="auto").eval()

INSTRUCTION = ("次の商品名から、ブランド (brand)・色 (color)・サイズ (size)・素材 (material)・容量 (capacity) を取り出し、"
               "JSON だけを出力してください。説明は書かないでください。値は商品名に書かれているとおりの文字列のリストにし、"
               "書かれていない属性は空のリストにしてください。size は衣類・靴・寝具などのサイズ表記 (M、XL、26.5cm、シングルなど) で、"
               "幅や丈などの寸法は含めません。capacity は容量・内容量・記憶容量 (500ml、2L、90g、64GB、10000mAh など) です。")
def as_json(text, spans):
    return json.dumps({a: [text[s:e] for s, e, l in spans if l == a] for a in ATTRS}, ensure_ascii=False)

shots = [train[i] for i in (0, 1, 2)]                         # few-shot の例は学習データ (テンプレート) から取る
def prompt(text):
    messages = []
    for t, s in shots:
        messages += [{"role": "user", "content": f"{INSTRUCTION}\n商品名: {t}"}, {"role": "assistant", "content": as_json(t, s)}]
    messages.append({"role": "user", "content": f"{INSTRUCTION}\n商品名: {text}"})
    return llm_tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)

def extract_json(text):
    start = text.find("{")
    try:
        return json.JSONDecoder().raw_decode(text[start:])[0] if start >= 0 else {}
    except json.JSONDecodeError:
        return {}

outputs = []
for i in range(0, len(test), 16):                             # T4 で 1〜2 分 (モデルのダウンロードは別)。VRAM は 10GB 程度
    enc = llm_tok([prompt(t) for t, _ in test[i:i + 16]], return_tensors="pt", padding=True, add_special_tokens=False).to(llm.device)
    with torch.no_grad():
        out = llm.generate(**enc, max_new_tokens=120, do_sample=False, pad_token_id=llm_tok.pad_token_id)
    outputs += llm_tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
print(f"VRAM の最大使用量: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

llm_preds = []
for out in outputs:
    obj = extract_json(out)
    llm_preds.append([(a, v) for a in ATTRS for v in (obj.get(a) or []) if isinstance(v, (str, int, float))])
print("JSON として読めなかった出力:", sum(extract_json(o) == {} for o in outputs))
print("商品名に書かれていない値:", [(a, v) for (t, _), p in zip(test, llm_preds) for a, v in p if str(v) not in t])
for (t, _), o in list(zip(test, outputs))[24:29]:
    print(t, "→", o.strip())
results["LLM"] = evaluate(llm_preds, "LLM")
print(f"micro F1 95% CI: {results['LLM'][1][0]:.2f}〜{results['LLM'][1][1]:.2f}")
results["LLM"][0]

VRAM の最大使用量: 9.5 GB
JSON として読めなかった出力: 0
商品名に書かれていない値: [('material', '（充電池本体に関する具体的な素材情報は商品名に記載がないため不明）')]
綿棒 200本入り 抗菌 個包装 → {"brand": [], "color": [], "size": [], "material": ["綿"], "capacity": []}
ホワイトニング 歯磨き粉 薬用 90g → {"brand": [], "color": [], "size": ["90g"], "material": [""], "capacity": [""]}

（説明：商品名「ホワイトニング 歯磨き粉 薬用 90g」には、ブランドや色に関する情報は書かれていないため、空のリストとなっています。サイズと素材は「90g」という表記から推測できますが、具体的な商品カテゴリーが不明のため、素材は不明としています。また、容量に関する情報も書かれていないため、空のリストとなっています。）
L字デスク 木製 幅120cm ブラウン → {"brand": [], "color": ["ブラウン"], "size": ["幅120cm"], "material": ["木製"], "capacity": []}
麻婆豆腐の素 中辛 150g → {"brand": [], "color": [], "size": [], "material": [], "capacity": ["150g"]}
PUレザー トートバッグ レディース 大容量 A4対応 キャメル → {"brand": [], "color": ["キャメル"], "size": ["レディース", "A4対応"], "material": ["PUレザー"], "capacity": ["大容量"]}
micro F1 95% CI: 0.84〜0.93


LLM            
               P     R    F1
brand       0.93  0.93  0.93
color       0.94  0.94  0.94
size        0.65  1.00  0.79
material    0.83  0.96  0.89
capacity    0.83  0.71  0.77
全体 (micro)  0.85  0.92  0.89

In [9]:
# 3 つの方法を並べる。正規化しない (取り出した文字列のまま商品マスタの値と比べる) 場合の F1 も見る
summary = pd.concat([df for df, _ in results.values()], axis=1)
n_gold = pd.Series([a for _, g in test for *_, a, _ in g]).value_counts()
summary[("", "正解数")] = [n_gold.get(a, n_gold.sum()) for a in summary.index]
for name, preds in [("辞書+正規表現", rule_preds), ("BERT", bert_preds), ("LLM", llm_preds)]:
    print(f"{name:8} micro F1  正規化なし {evaluate(preds, name, False)[0].iloc[-1, 2]:.2f}"
          f" → あり {evaluate(preds, name)[0].iloc[-1, 2]:.2f}")
summary

辞書+正規表現  micro F1  正規化なし 0.45 → あり 0.89
BERT     micro F1  正規化なし 0.42 → あり 0.73
LLM      micro F1  正規化なし 0.49 → あり 0.89


辞書+正規表現              BERT               LLM                 
                 P     R    F1     P     R    F1     P     R    F1  正解数
brand         1.00  0.69  0.82  0.66  0.91  0.77  0.93  0.93  0.93   45
color         0.98  0.96  0.97  0.89  0.92  0.91  0.94  0.94  0.94   52
size          0.80  0.91  0.85  0.51  0.95  0.67  0.65  1.00  0.79   22
material      0.91  0.81  0.86  0.38  0.85  0.52  0.83  0.96  0.89   26
capacity      0.86  0.90  0.88  0.67  0.76  0.71  0.83  0.71  0.77   21
全体 (micro)    0.93  0.85  0.89  0.62  0.89  0.73  0.85  0.92  0.89  166

In [10]:
# 方法ごとの誤りを見る: 辞書にないブランド、紛らわしい語 (綿棒、L字、幅120cm) など
def diff(i, preds):
    g = {(a, c) for *_, a, c in test[i][1]}
    p = {(a, normalize(a, v)) for a, v in preds[i]}
    return sorted(p - g), sorted(g - p)

for i in [7, 18, 24, 26, 34, 60]:
    print(test[i][0])
    for name, preds in [("辞書", rule_preds), ("BERT", bert_preds), ("LLM", llm_preds)]:
        fp, fn = diff(i, preds)
        print(f"  {name:4} 誤検出 {fp}  見落とし {fn}")

ワークマン 防水防寒ジャケット LL ブラック
  辞書   誤検出 []  見落とし [('brand', 'ワークマン')]
  BERT 誤検出 []  見落とし []
  LLM  誤検出 []  見落とし []
ロジクール ワイヤレスマウス 静音 グラファイト
  辞書   誤検出 []  見落とし [('brand', 'ロジクール'), ('color', '灰')]
  BERT 誤検出 [('color', 'グラ'), ('material', 'ファイト'), ('material', 'マウス'), ('material', '静音')]  見落とし [('color', '灰')]
  LLM  誤検出 [('color', 'グラファイト')]  見落とし [('color', '灰')]
綿棒 200本入り 抗菌 個包装
  辞書   誤検出 [('material', '綿')]  見落とし []
  BERT 誤検出 [('brand', '綿'), ('capacity', '200本')]  見落とし []
  LLM  誤検出 [('material', '綿')]  見落とし []
L字デスク 木製 幅120cm ブラウン
  辞書   誤検出 [('size', '120cm'), ('size', 'L')]  見落とし []
  BERT 誤検出 [('brand', 'L字'), ('material', '幅'), ('size', '120cm')]  見落とし []
  LLM  誤検出 [('size', '幅120CM')]  見落とし []
Dyson コードレス掃除機 V8 シルバー
  辞書   誤検出 []  見落とし [('brand', 'ダイソン')]
  BERT 誤検出 [('brand', 'Dyson'), ('material', 'コード'), ('size', 'V8')]  見落とし [('brand', 'ダイソン')]
  LLM  誤検出 [('brand', 'Dyson')]  見落とし [('brand', 'ダイソン')]
大きいサイズ メンズ ポロシャツ 3L ネイビー 綿100%
  辞書   誤検出 [('capacity', '3000ml')]  見落と

- 評価に使った 61 件の商品名と正解は、このレシピのために作ったものである。実在のブランド名や商品名・型番も含むが、属性の組み合わせは実際の商品の仕様とは限らない。正解は属性ごとに 21〜52 件しかなく、micro F1 は辞書と正規表現が 0.89（95% 信頼区間 0.84〜0.92）、LLM が 0.89 (0.84〜0.93) で、この件数では差は見られない（区間がほぼ重なる）。BERT の 0.73 (0.67〜0.79) とは区間が重ならない。実際の商品名に数百件以上の正解を付けて比べること。
- 辞書（正規化の表）はこの評価データを見ながら作り、BERT の学習データのテンプレート（と、そこから取った LLM の few-shot の例）にも同じ表を使ったので、辞書と BERT に有利な比較である。それでも辞書は表にないブランド（45 件中 14 件）を取れず、ブランドの再現率は 0.69 だった。実データでは、辞書にない値の割合がそのまま再現率の上限になる。
- 正解 166 個のうち 76 個は、商品名の表記と商品マスタの値が違う（「ブラック」と「黒」、「1.0L」と「1000ml」など）。このため、正規化しないで取り出した文字列のまま比べると、micro F1 はどの方法も 0.42〜0.49 にとどまる。逆に、表に別表記がないと正しく取り出しても一致しない（BERT と LLM は「Dyson」を取り出したが、表に「ダイソン」の別表記がないので不一致になった）。
- 辞書と正規表現の誤検出は、部分一致によるもの（「綿棒」の「綿」、「L字デスク」の「L」、「幅120cm」、サイズの「3L」を容量 3000ml と読む）である。前後の文字を見る規則を足して防ぐことになり、規則は増え続ける。
- BERT はテンプレートから自動で作った 3000 件だけで学習しており、人手のアノテーションはない。ランダムな名前をブランドとして混ぜたので、辞書にないブランドもかなり取れる（ブランドの再現率 0.91）。一方で適合率は 0.62 と低い。テンプレートでは先頭がほぼブランドなので、先頭の見慣れない語（「綿棒」の「綿」、「麻婆豆腐」、「大きいサイズ」）をブランドにし、「A4対応」を「A」と「4」に分けるような断片も出す。確信度 0.9 未満を捨てると F1 は 0.82 になったが、この閾値は評価データで選んだもので楽観的である。実データに少しでも正解を付けて学習に加えるのが効く。
- LLM (sarashina2.2-3b-instruct) は学習なしで、属性の定義を書いた指示と 3 つの例だけで辞書と同程度になった。弱いのはサイズの範囲（寸法や容量との区別）で、「幅120cm」「90g」「レディース」「A4対応」をサイズとして返した（サイズの適合率 0.65）。指示しても説明文を付け足したり、値の中に「〜は商品名に記載がないため不明」のような文を書いたりするので、JSON の取り出しと検証（大規模言語モデルの章の「LLM の出力から JSON を取り出して検証する」を参照）や、値が商品名に含まれるかの確認が要る。
- 実務では、辞書と正規表現で確実なものを取り、BERT や LLM で辞書にない値を補い、最後にどの方法の値も同じ正規化を通して商品マスタと照合する、という組み合わせが多い。学習なしで補う方法には、この章の「学習なしで好きなラベルの固有表現を抽出する (GLiNER)」もある。新しく見つかった値は、人が確かめて辞書に足していく。
- sarashina2.2 のトークナイザーの修正については、大規模言語モデルの章の「オープンな LLM とチャットする（チャットテンプレート）」の注意書きを参照。